<!-- NB_DOC_INTRO_V1 -->
# Polars — Cheat-sheet exhaustive

> 📚 **Doc thematique** : [docs/01_STRUCTURES.md](docs/01_STRUCTURES.md) (Structures Python / DataFrame / Preprocessing)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Polars** (https://pola.rs) est une bibliotheque DataFrame ecrite en **Rust** avec bindings Python, concue pour remplacer / completer pandas sur les gros volumes (10^6 - 10^9 lignes).
Caracteristiques clefs : **multi-thread par defaut** (utilise tous les cores), **lazy evaluation optionnelle** (optimiseur de requete), **API expression-based** (immutable, composable), **Apache Arrow** comme format en memoire (interop gratuit avec pandas, DuckDB, PyArrow).

Ce notebook couvre l'API en profondeur : creation, selection, filtrage, expressions, group_by, joins (avec `join_asof`), reshape (pivot/melt/stack), datetime, strings, window functions, lazy mode + optimiseur, streaming, IO (Parquet/CSV/JSON), interop pandas, et pieges courants. Versions : `polars >= 0.20` (2024), API stable pour ce notebook.

**Quand utiliser Polars vs pandas vs DuckDB :**
- **pandas** : < 1M lignes, ecosystème ML existant
- **Polars** : 1M - 100M lignes en RAM, expressions fortes, API moderne
- **DuckDB** : SQL natif, dataset gros (> 100M), Parquet/Arrow direct

## Plan

1. Installation et imports
2. Creation de DataFrames (eager vs lazy)
3. Inspection et types
4. Selection / filtrage avec expressions
5. GroupBy et agregations
6. Joins (inner, outer, left, semi, anti, asof, cross)
7. Reshape (pivot, melt, stack, unstack)
8. Datetime (parsing, accessors, timezone, resample/group_by_dynamic)
9. Strings (str. namespace)
10. Window functions (over)
11. Lazy mode et optimiseur (predicate pushdown, projection pushdown)
12. Streaming mode (datasets > RAM)
13. IO (CSV, Parquet, JSON, Arrow)
14. Interop avec pandas et numpy
15. Comparatifs perf vs pandas
16. Pieges et anti-patterns
17. References


## 1. Installation et imports

Polars s'installe avec une seule dependance principale (`pyarrow` est inclus pour les operations IO/interop).


In [ ]:
# pip install -q polars pyarrow
import polars as pl
import numpy as np
import pandas as pd
from datetime import datetime, date, timedelta

# Setup reproductibilite
SEED = 42
np.random.seed(SEED)

# Affichage compact
pl.Config.set_tbl_rows(8)
pl.Config.set_tbl_cols(10)

print(f"Polars version : {pl.__version__}")
print(f"Pandas version : {pd.__version__}")

## 2. Creation de DataFrames

Polars supporte plusieurs sources : dict Python, numpy arrays, listes, pandas DataFrame, fichiers, queries SQL via Arrow.

### 2.1 Depuis un dict Python (cas le plus courant)


In [ ]:
df = pl.DataFrame({
    "id":       [1, 2, 3, 4, 5],
    "name":     ["Alice", "Bob", "Charlie", "Dave", "Eve"],
    "score":    [85.5, 72.1, 90.3, 68.7, 91.0],
    "category": ["A", "B", "A", "C", "B"],
    "active":   [True, True, False, True, False],
})
df

**Lecture du resultat** : les types sont **typés** strictement (`Int64`, `Utf8`, `Float64`, `Boolean`). Polars infère depuis Python mais on peut forcer les types via `schema=`.

### 2.2 Avec schema explicite (recommande en production)


In [ ]:
df_typed = pl.DataFrame(
    {
        "id":       [1, 2, 3],
        "amount":   [100, 200, 300],
        "currency": ["EUR", "USD", "EUR"],
    },
    schema={
        "id":       pl.UInt32,        # entier non signe (suffit pour des ids)
        "amount":   pl.Float64,
        "currency": pl.Categorical,   # ~ category pandas, optimise pour cardinalite faible
    },
)
print(df_typed.dtypes)
df_typed

### 2.3 Depuis numpy


In [ ]:
arr = np.random.randn(5, 3)
df_np = pl.DataFrame(arr, schema=["a", "b", "c"])
df_np

### 2.4 Depuis pandas (interop)


In [ ]:
pd_df = pd.DataFrame({"x": [1, 2, 3], "y": [10.0, 20.0, 30.0]})
df_from_pd = pl.from_pandas(pd_df)
df_from_pd

### 2.5 Eager vs Lazy

- **`pl.DataFrame`** = mode **eager** : execute immediatement, comme pandas.
- **`pl.LazyFrame`** = mode **lazy** : differe l'execution, permet l'optimiseur (predicate / projection pushdown, fusion d'operations).

> 🎯 **Regle** : utiliser **lazy** pour les pipelines analytiques (filter → group → agg → sort), et **eager** pour l'exploration interactive.


In [ ]:
# Lazy : on construit un PLAN, rien n'est execute
ldf = pl.LazyFrame({"x": range(1000), "y": range(1000)})
plan = (
    ldf
    .filter(pl.col("x") > 500)
    .with_columns((pl.col("y") * 2).alias("y2"))
    .select(["x", "y2"])
)
print(plan.explain())   # affiche le plan optimise

In [ ]:
# Pour materialiser : .collect()
result = plan.collect()
print(result.head())

## 3. Inspection et types

Connaitre les types et la forme avant de transformer = ne pas se faire avoir par un `Int64` au lieu de `Float64`.


In [ ]:
print(f"Shape           : {df.shape}")
print(f"Nb lignes       : {df.height}")
print(f"Nb colonnes     : {df.width}")
print(f"Schema (types)  : {df.schema}")
print(f"Memory (bytes)  : {df.estimated_size()}")
print()
df.describe()

### 3.1 Cast de types


In [ ]:
df_cast = df.with_columns([
    pl.col("score").cast(pl.Int32).alias("score_int"),     # float -> int (tronque)
    pl.col("category").cast(pl.Categorical),               # str -> categorical (gain memoire)
])
print(df_cast.dtypes)
df_cast

## 4. Selection et filtrage avec **expressions**

L'**expression** est le concept central de Polars. `pl.col("x")` est une expression deferee qui sera evaluee dans un contexte (`select`, `filter`, `with_columns`, `group_by(...).agg(...)`).

Les expressions sont **immuables et composables** : `(pl.col("x") + 1).alias("x1")` ne modifie pas `x`, mais cree une nouvelle expression.

### 4.1 Selection de colonnes


In [ ]:
# Liste de colonnes (par nom)
df.select(["name", "score"])

In [ ]:
# Expressions complexes dans select
df.select(
    pl.col("name"),
    (pl.col("score") * 1.1).alias("score_plus10pct"),       # nouvelle colonne calculee
    pl.col("score").max().alias("max_score"),                # scalaire broadcast a toutes lignes
    pl.lit(2024).alias("year"),                              # litteral
)

### 4.2 Selection par dtype ou pattern


In [ ]:
# Toutes les colonnes numeriques
df.select(pl.col(pl.NUMERIC_DTYPES))

In [ ]:
# Toutes les colonnes commencant par "s"
df.select(pl.col("^s.*$"))   # regex

### 4.3 Filtrage (`filter` ~ `df[df['x'] > 5]` de pandas)

Operateurs : `&` (et), `|` (ou), `~` (non). **Toujours parentheser**.


In [ ]:
df.filter(pl.col("score") > 80)

In [ ]:
# Multi-conditions avec & et |
df.filter(
    (pl.col("score") > 70) & (pl.col("category").is_in(["A", "B"]))
)

In [ ]:
# Negation avec ~
df.filter(~pl.col("active"))    # ou : pl.col("active").not_()

In [ ]:
# is_null / is_not_null / is_between
df_with_nulls = pl.DataFrame({"x": [1, 2, None, 4, None], "y": [10, 20, 30, None, 50]})
print(df_with_nulls.filter(pl.col("x").is_not_null()))

## 5. GroupBy et agregations

Notez **`group_by`** (avec underscore, depuis 0.19) au lieu de `groupby` pandas.


In [ ]:
# Donnees plus interessantes
np.random.seed(SEED)
sales = pl.DataFrame({
    "region":   np.random.choice(["N", "S", "E", "W"], 1000),
    "category": np.random.choice(["A", "B", "C"], 1000),
    "amount":   np.random.exponential(100, 1000),
    "qty":      np.random.poisson(5, 1000),
    "date":     pl.date_range(date(2024, 1, 1), date(2024, 1, 1) + timedelta(days=999),
                              interval="1d", eager=True)[:1000],
})
print(sales.head())

### 5.1 Agregations multiples par groupe


In [ ]:
(
    sales
    .group_by("region")
    .agg(
        pl.col("amount").sum().alias("total"),
        pl.col("amount").mean().alias("mean"),
        pl.col("amount").median().alias("median"),
        pl.col("amount").std().alias("std"),
        pl.col("amount").quantile(0.95).alias("p95"),
        pl.col("qty").sum().alias("qty_total"),
        pl.len().alias("n_orders"),                    # count = pl.len()
    )
    .sort("total", descending=True)
)

### 5.2 GroupBy multi-cles + condition


In [ ]:
(
    sales
    .group_by(["region", "category"])
    .agg(
        pl.col("amount").sum().alias("total"),
        # Conditional aggregation : ne sommer que les amount > 50
        pl.col("amount").filter(pl.col("amount") > 50).sum().alias("total_above_50"),
        # Compter les distincts
        pl.col("category").n_unique().alias("n_unique_cat"),   # ici = 1, mais demo
    )
    .sort(["region", "category"])
    .head(8)
)

## 6. Joins

API simple : `df1.join(df2, on='id', how='inner')`. Polars supporte aussi des joins specialises tres utiles : `asof` (TS), `semi`, `anti`, `cross`.


In [ ]:
orders = pl.DataFrame({
    "order_id":    [1, 2, 3, 4],
    "customer_id": [10, 20, 10, 30],
    "amount":      [100, 200, 150, 300],
})
customers = pl.DataFrame({
    "customer_id":  [10, 20, 40],
    "name":         ["Alice", "Bob", "Frank"],
    "city":         ["Paris", "Lyon", "Marseille"],
})

# Inner join : seuls les customers presents dans LES DEUX
print("INNER :")
print(orders.join(customers, on="customer_id"))

In [ ]:
# Left join : garde toutes les commandes, name/city NULL si pas trouve
print("LEFT :")
print(orders.join(customers, on="customer_id", how="left"))

In [ ]:
# Anti join : commandes SANS customer connu (utile pour data quality)
print("ANTI :")
print(orders.join(customers, on="customer_id", how="anti"))

In [ ]:
# Semi join : commandes AVEC customer connu (sans ramener les colonnes de customer)
print("SEMI :")
print(orders.join(customers, on="customer_id", how="semi"))

### 6.1 `join_asof` — jointure "la plus proche par le bas" (TS)

Cas d'usage classique : pour chaque trade, recuperer le **dernier prix connu**.


In [ ]:
prices = pl.DataFrame({
    "ts":    [datetime(2024, 1, 1, 9, 0, 0),
              datetime(2024, 1, 1, 9, 1, 0),
              datetime(2024, 1, 1, 9, 2, 0),
              datetime(2024, 1, 1, 9, 5, 0)],
    "price": [100.0, 101.5, 102.3, 105.0],
})
trades = pl.DataFrame({
    "ts":  [datetime(2024, 1, 1, 9, 0, 30),
            datetime(2024, 1, 1, 9, 1, 45),
            datetime(2024, 1, 1, 9, 4, 0),
            datetime(2024, 1, 1, 9, 10, 0)],
    "qty": [10, 5, 20, 8],
})

# Recuperer le prix le plus proche AVANT chaque trade (strategy='backward')
trades.sort("ts").join_asof(prices.sort("ts"), on="ts", strategy="backward")

## 7. Reshape (pivot, melt, stack)

### 7.1 pivot (long → wide)


In [ ]:
long = pl.DataFrame({
    "date":     ["2024-01", "2024-01", "2024-02", "2024-02"],
    "product":  ["A", "B", "A", "B"],
    "sales":    [100, 200, 150, 250],
})
wide = long.pivot(values="sales", index="date", on="product", aggregate_function="sum")
wide

### 7.2 melt / unpivot (wide → long)


In [ ]:
long_again = wide.unpivot(index="date", variable_name="product", value_name="sales")
long_again

## 8. Datetime — accessors `dt.`

Polars a un **type Date** (pas de time component) et un **Datetime** (avec time, ms precision par defaut).
API namespace `pl.col("d").dt.{year, month, day, hour, weekday, quarter, ...}`.


In [ ]:
ts_df = pl.DataFrame({
    "date": pl.date_range(date(2024, 1, 1), date(2024, 12, 31), interval="1d", eager=True),
})
ts_df = ts_df.with_columns(
    pl.col("date").dt.year().alias("year"),
    pl.col("date").dt.month().alias("month"),
    pl.col("date").dt.day().alias("day"),
    pl.col("date").dt.weekday().alias("dow"),         # 1=lundi, 7=dimanche
    pl.col("date").dt.week().alias("iso_week"),
    pl.col("date").dt.quarter().alias("quarter"),
    pl.col("date").dt.ordinal_day().alias("doy"),     # 1-366
)
ts_df.head()

### 8.1 Parsing dates (depuis string)


In [ ]:
parsed = pl.DataFrame({"s": ["2024-01-15", "2024-02-29", "invalid"]})
parsed = parsed.with_columns(
    pl.col("s").str.to_date(format="%Y-%m-%d", strict=False).alias("d_strict_false"),
)
parsed   # invalid -> null (strict=False)

### 8.2 Resample / aggregation temporelle avec `group_by_dynamic`

Equivalent du `resample` pandas, mais plus puissant (fenetre glissante, by group).


In [ ]:
sales_daily = sales.sort("date").group_by_dynamic("date", every="1mo").agg(
    pl.col("amount").sum().alias("monthly_total"),
    pl.col("qty").sum().alias("monthly_qty"),
)
sales_daily.head()

### 8.3 Rolling window


In [ ]:
sales_with_roll = sales.sort("date").with_columns(
    pl.col("amount").rolling_mean(window_size=7).alias("rolling_7d_mean"),
    pl.col("amount").rolling_sum(window_size=30).alias("rolling_30d_sum"),
)
sales_with_roll.tail()

## 9. String operations — namespace `str.`

Toutes les ops sur strings passent par `.str.`. Equivalents de pandas mais souvent plus rapides (vectorise Rust).


In [ ]:
txt = pl.DataFrame({"email": ["alice@gmail.com", "BOB@yahoo.fr", "Charlie@hotmail.COM", "invalid"]})

txt.with_columns(
    pl.col("email").str.to_lowercase().alias("email_low"),
    pl.col("email").str.contains("gmail").alias("is_gmail"),
    pl.col("email").str.split("@").list.get(1, null_on_oob=True).alias("domain"),  # null si pas de @
    pl.col("email").str.extract(r"^([^@]+)@", group_index=1).alias("local_part"),
    pl.col("email").str.len_chars().alias("len_chars"),
    pl.col("email").str.replace("@", " AT ").alias("safe"),
)

## 10. Window functions avec `over`

`expr.over(partition)` = equivalent pandas `groupby().transform()`. Calcule l'expression par groupe **sans agreger** (le resultat a la meme taille que l'input).


In [ ]:
(
    sales.with_columns(
        # Total par region (sans agreger)
        pl.col("amount").sum().over("region").alias("region_total"),
        # Part dans la region
        (pl.col("amount") / pl.col("amount").sum().over("region")).alias("share_in_region"),
        # Rang dans la region (1 = max)
        pl.col("amount").rank(method="dense", descending=True).over("region").alias("rank_in_region"),
        # Cumulative dans la region par date
        pl.col("amount").cum_sum().over("region").alias("cumsum_region"),
    )
    .head(8)
)

## 11. Lazy mode + optimiseur

L'interet principal du mode lazy est l'**optimiseur** : Polars peut **reordonner / fusionner** les operations. Exemple : `filter` apres `join` → l'optimiseur le **pousse avant le join** (predicate pushdown) → on joine moins de lignes.

### Optimisations clefs
- **Predicate pushdown** : filtres pousses au plus tot (en bas du graphe d'execution)
- **Projection pushdown** : ne lit/calcule que les colonnes necessaires (gigantesque sur Parquet)
- **Common subexpression elimination** : evite de recalculer une expression utilisee plusieurs fois
- **Slice pushdown** : pour les `.head(n)`, ne materialise que `n` lignes


In [ ]:
# Construire un plan complexe
ldf = (
    pl.LazyFrame({
        "id":   list(range(10_000)),
        "x":    np.random.randn(10_000),
        "y":    np.random.randn(10_000),
        "grp":  np.random.choice(["A","B","C","D"], 10_000),
    })
    .with_columns((pl.col("x") + pl.col("y")).alias("sum_xy"))
    .filter(pl.col("x") > 0)                       # filter applique apres
    .group_by("grp")
    .agg(
        pl.col("x").mean(),
        pl.col("y").mean(),
        pl.col("sum_xy").max(),
    )
    .sort("grp")
)

# Plan non optimise
print("=== NAIVE PLAN ===")
print(ldf.explain(optimized=False))

In [ ]:
# Plan optimise (l'optimiseur a pousse le filter avant la with_columns,
# fusionne les operations, etc.)
print("=== OPTIMIZED PLAN ===")
print(ldf.explain(optimized=True))

In [ ]:
# Executer
ldf.collect()

## 12. Streaming mode (datasets > RAM)

Pour traiter un fichier **plus gros que la RAM**, on combine `scan_*` (lazy IO) + `.collect(streaming=True)`. Polars chunke automatiquement.


In [ ]:
# Demo : ecrire un gros parquet et le lire en streaming
import tempfile, os

# Generer ~ 1M lignes et ecrire en parquet
big_df = pl.DataFrame({
    "id":  np.arange(1_000_000),
    "val": np.random.randn(1_000_000),
    "grp": np.random.choice(["X", "Y", "Z"], 1_000_000),
})

tmpfile = os.path.join(tempfile.gettempdir(), "polars_demo.parquet")
big_df.write_parquet(tmpfile)
print(f"Fichier ecrit : {os.path.getsize(tmpfile) / 1024**2:.1f} MB")

# Lecture en streaming
result = (
    pl.scan_parquet(tmpfile)        # rien charge en RAM
    .filter(pl.col("val") > 0)
    .group_by("grp")
    .agg(pl.col("val").mean().alias("mean_pos"), pl.len().alias("n"))
    .collect(streaming=True)
)
print(result)

os.remove(tmpfile)

## 13. IO — formats supportes

| Format | Lecture eager | Lecture lazy | Ecriture | Quand |
|---|---|---|---|---|
| CSV  | `read_csv` | `scan_csv` | `write_csv` | Echange basique. Lent. |
| **Parquet** | `read_parquet` | `scan_parquet` | `write_parquet` | **Recommande** : columnar, compressed, schema |
| JSON | `read_json` | (scan via NDJSON) | `write_json` / `write_ndjson` | API web |
| Excel | `read_excel` | - | `write_excel` | Si on a pas le choix |
| Arrow IPC | `read_ipc` | `scan_ipc` | `write_ipc` | Interop interne ultra-rapide |
| Avro | `read_avro` | - | `write_avro` | Pipelines Kafka |
| ODS | `read_ods` | - | - | LibreOffice |
| Database | `read_database` | - | `write_database` | via connectorx/SQLAlchemy |

> 🎯 **Best practice** : convertir CSV en Parquet une seule fois, puis travailler en Parquet. Gain typique : x10 vitesse, /5 taille.


In [ ]:
# Demo Parquet : ecriture + lecture
import io

# Ecrire en memoire (BytesIO) pour la demo
buf = io.BytesIO()
df.write_parquet(buf)
print(f"Taille Parquet (5 lignes) : {len(buf.getvalue())} bytes")

# Lire
buf.seek(0)
df_back = pl.read_parquet(buf)
df_back.equals(df)

## 14. Interop avec pandas et numpy


In [ ]:
# Polars -> pandas (avec backend pyarrow recommande, plus rapide)
pd_back = df.to_pandas(use_pyarrow_extension_array=True)
print("Pandas dtypes (pyarrow) :")
print(pd_back.dtypes)
pd_back

In [ ]:
# Polars -> numpy
arr = df.select("score").to_numpy()
print(f"Numpy shape : {arr.shape}, dtype : {arr.dtype}")

# Pandas -> Polars
pl.from_pandas(pd_back).head()

### 14.1 Avec DuckDB (SQL natif sur DataFrame Polars)

DuckDB peut requeter un DataFrame Polars **sans copie** grace a Arrow.


In [ ]:
try:
    import duckdb
    duckdb.sql("SELECT category, AVG(score) AS mean_score FROM df GROUP BY category").pl()
except ImportError:
    print("DuckDB pas installe (uv add --group bdd duckdb)")

## 15. Comparatif perf vs pandas

Benchmark simple sur 1M lignes : groupby + aggregation.


In [ ]:
import time

n = 1_000_000
data = {
    "grp": np.random.choice(["A", "B", "C", "D"], n),
    "val": np.random.randn(n),
    "qty": np.random.randint(1, 100, n),
}

# Pandas
t0 = time.perf_counter()
pd_df = pd.DataFrame(data)
res_pd = pd_df.groupby("grp").agg(
    mean=("val", "mean"),
    sum_qty=("qty", "sum"),
    n=("val", "count"),
)
t_pd = time.perf_counter() - t0

# Polars eager
t0 = time.perf_counter()
pl_df = pl.DataFrame(data)
res_pl = pl_df.group_by("grp").agg(
    pl.col("val").mean().alias("mean"),
    pl.col("qty").sum().alias("sum_qty"),
    pl.len().alias("n"),
)
t_pl = time.perf_counter() - t0

# Polars lazy
t0 = time.perf_counter()
res_pl_lazy = (
    pl.LazyFrame(data)
    .group_by("grp")
    .agg(pl.col("val").mean().alias("mean"),
         pl.col("qty").sum().alias("sum_qty"),
         pl.len().alias("n"))
    .collect()
)
t_pl_lazy = time.perf_counter() - t0

print(f"Pandas     : {t_pd*1000:7.1f} ms")
print(f"Polars     : {t_pl*1000:7.1f} ms  ({t_pd/t_pl:.1f}x vs pandas)")
print(f"Polars lazy: {t_pl_lazy*1000:7.1f} ms  ({t_pd/t_pl_lazy:.1f}x vs pandas)")

## 16. Pieges et anti-patterns

| ❌ Anti-pattern Polars | ✅ Correctif |
|---|---|
| Iterer ligne par ligne (`for row in df.iter_rows()`) | Toujours utiliser des expressions vectorisees |
| `df["col"] = ...` (mutation pandas-style) | `df = df.with_columns(...)` (immutable) |
| `and`, `or`, `not` dans `filter` | `&`, `|`, `~` avec parentheses |
| Lire un gros CSV en eager | `pl.scan_csv("file.csv")` (lazy) |
| `.collect()` plusieurs fois sur la meme query | Mettre en cache : `df_cached = lazy.collect().lazy()` |
| Garder CSV pour stockage > 100 MB | Convertir en Parquet (×10 vitesse, /5 taille) |
| `groupby` (sans underscore) | `group_by` (l'API a change en 0.19) |
| Convertir tout en pandas pour "lire" | Polars a son propre repr (`print(df)` ou last expr) |
| `df.apply(func)` sur scalaires | Utiliser une expression native si possible |
| Comparer dtypes pandas et polars naivement | Conversion explicite ou via pyarrow |

### Erreurs frequentes


In [ ]:
# ❌ FAUX : oublier les parentheses
# df.filter(pl.col("score") > 50 & pl.col("active"))   # ERREUR

# ✅ BON : parentheses
df.filter((pl.col("score") > 50) & pl.col("active"))

In [ ]:
# ❌ FAUX : tenter une mutation pandas-style
# df["new_col"] = df["score"] * 2   # ERREUR : DataFrame n'a pas __setitem__

# ✅ BON : with_columns (immutable, retourne un nouveau DF)
df_new = df.with_columns((pl.col("score") * 2).alias("score_x2"))
df_new

## 17. References

### Documentation officielle
- User guide : https://docs.pola.rs/user-guide/
- API reference : https://docs.pola.rs/py-polars/html/reference/
- Migration depuis pandas : https://docs.pola.rs/user-guide/migration/pandas/

### Benchmarks publics
- DuckDB / Polars / pandas H2O benchmarks : https://duckdblabs.github.io/db-benchmark/
- TPCH-like benchmarks (Polars repo) : https://github.com/pola-rs/polars-book/blob/main/user_guide/benchmarks/

### Tutoriels et blogs
- Modern Polars (book gratuit) : https://kevinheavey.github.io/modern-polars/
- Polars vs Pandas examples : https://github.com/jorisvandenbossche/talk-polars-vs-pandas

### Voir aussi (collection)
- [Structures_DataFrame.ipynb](./Structures_DataFrame.ipynb) — equivalent pandas exhaustif
- [BDD_DuckDB.ipynb](./BDD_DuckDB.ipynb) — DuckDB peut requeter un DF Polars en SQL
- [DE_PySpark.ipynb](./DE_PySpark.ipynb) — pour datasets > 1 TB ou multi-node
- [DS_Metrics_Reference.ipynb](./DS_Metrics_Reference.ipynb) — metriques d'evaluation ML
